# Moment scaling

This notebook performs the moment scaling analysis of individual seismic acceleration signals, following the framework of Vollmer et al. (2024). Results from the PDF 
analysis (notebook 03a) are imported at the end to build the complete summary table.

## 1. Imports and visualization settings

In [ ]:
from pathlib import Path
import pandas as pd
import logging
from src import (
    integrate_to_velocity,
    integrate_to_displacement,
    compute_moment_scaling,
    compute_increments,
    compute_moments_from_increments,
    compute_scaling_exponents,
    test_scaling_linearity,
    fit_piecewise_scaling,
    trim_to_event_window,
    build_scaling_summary,
    set_plot_style,
    plot_onset_diagnostic,
    plot_onset_distribution,
    plot_increments_histograms_dual_view
)
colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

INFO | Environment ready


## 2. Data loading

In [3]:
logger.info("Loading preprocessed data...")
df_acc_clean = pd.read_parquet('../data/processed/02_signals/acc_preprocessed_pdf.parquet')
df_acc_long  = pd.read_parquet('../data/processed/02_signals/acc_preprocessed_scaling.parquet')
check(len(df_acc_clean) > 0, f"All signals loaded: {df_acc_clean['file'].nunique()} files")
check(len(df_acc_long) > 0,  f"Long signals loaded: {df_acc_long['file'].nunique()} files")

FIGURES_DIR = Path('../figures/03_single_signal/03b_moment_scaling')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / 'scaling' / 'displacement' / 'event_window').mkdir(parents=True, exist_ok=True)
check(FIGURES_DIR.exists(), f"Figures directory ready: {FIGURES_DIR}")

INFO | Loading preprocessed data...
INFO | All signals loaded: 66 files
INFO | Long signals loaded: 48 files
INFO | Figures directory ready: ../figures/03_single_signal/03b_moment_scaling


## 3. Moment scaling analysis

The scaling behavior of statistical moments is investigated to detect
signatures of anomalous or strongly anomalous diffusion, following the
framework of Vollmer et al. (2024).

For a process $x(t)$, the displacement over a time scale $\tau$ starting
from $t_0$ is defined as:

$$\Delta x(\tau, t_0) = x(t_0 + \tau) - x(t_0)$$

The $q$-th order moment is estimated as a temporal average over all
possible starting points $t_0$ (sliding window):

$$M_q(\tau) = \langle |\Delta x(\tau, t_0)|^q \rangle_{t_0}
= \frac{1}{N - \tau} \sum_{t_0=0}^{N-\tau-1} |\Delta x(\tau, t_0)|^q$$

If the process exhibits scaling, the moments obey a power law in $\tau$:

$$M_q(\tau) \sim \tau^{\zeta(q)}$$

For normal diffusion, $\zeta(q) = q/2$ (linear in $q$). Strong anomalous
diffusion is characterized by a piecewise-linear spectrum with a crossover
at $q = \alpha$ (Vollmer et al., 2024, Eq.~1b):

$$\zeta(q) = \begin{cases} \xi\, q & q \leq \alpha \\ \zeta_0\, q - \alpha(\zeta_0 - \xi) & q > \alpha \end{cases}$$

Three definitions of the process $x(t)$ are explored and compared:

- **Section 4** — $x(t) = a(t)$: the acceleration signal itself; increments
  are differences of acceleration values $\Delta a(\tau, t_0) = a(t_0+\tau) - a(t_0)$.
- **Section 5** — $x(t) = v(t) = \sum_{k} a(k)\,\Delta t$: the ground velocity,
  obtained by integrating the acceleration once.
- **Section 6** — $x(t) = \int v(t)\,dt$: the ground displacement,
  obtained by integrating the acceleration twice.

All three versions use a sampling interval $\Delta t = 0.005$\,s (200\,Hz).
The 6 near-field stations with short recordings (SURF, BRZ, BHB, CRI, SLZ, SAV)
are excluded from all moment scaling analyses, leaving 48 signals.

### 3.1 Acceleration

#### 3.1.1 Computation of q-th order moments

The process is defined as $x(t) = a(t)$, the normalized acceleration signal.
Increments are computed as:

$$\Delta a(\tau, t_0) = a(t_0 + \tau) - a(t_0)$$

Moments $M_q(\tau)$ are computed for moment orders $q \in \{0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0\}$ and time scales $\tau \in \{10, 50, 100, 200, 500, 1000, 2000, 5000, 10000\}$ samples.

In [4]:
logger.info("Computing acceleration increments — full signal...")
q_values_acc = q_values = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0, 
                            3.25, 3.5, 3.75, 4.0, 4.25, 4.5, 4.75, 5.0]
tau_values_acc = [10, 20, 35, 50, 75, 100, 150, 200, 300, 500,
                  750, 1000, 1500, 2000, 3000, 5000, 7500, 10000]

# Compute increments
df_increments_acc = compute_increments(df_acc_long, tau_values_acc, column='acceleration')
logger.info(f"Increments computed: {df_increments_acc.shape}")

# Save increments
try:
    df_increments_acc.to_parquet('../data/processed/03b_moment_scaling/acceleration/full_signal/scaling_increments_acc.parquet', index=False)
    logger.info(f"Increments saved: {df_increments_acc.shape}")
except Exception as e:
    logger.error(f"Error saving increments: {e}")

# Compute moments
logger.info("Computing acceleration moments — full signal...")
df_moments_acc = compute_moments_from_increments(df_increments_acc, q_values_acc)
logger.info(f"Moments computed: {df_moments_acc.shape}")

# Save moments
try:
    df_moments_acc.to_parquet('../data/processed/03b_moment_scaling/acceleration/full_signal/scaling_moments_acc.parquet', index=False)
    logger.info(f"Moments saved: {df_moments_acc.shape}")
except Exception as e:
    logger.error(f"Error saving moments: {e}")

INFO | Computing acceleration increments — full signal...
INFO | Increments computed: (40358880, 6)
INFO | Increments saved: (40358880, 6)
INFO | Computing acceleration moments — full signal...
INFO | Moments computed: (16416, 6)
INFO | Moments saved: (16416, 6)


#### 3.1.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

In [5]:
logger.info("Computing scaling exponents — acceleration, full signal...")
df_exponents_acc = compute_scaling_exponents(
    df_moments_acc, output_dir=FIGURES_DIR / 'scaling' / 'acceleration' / 'exponents')
try:
    df_exponents_acc.to_parquet('../data/processed/03b_moment_scaling/acceleration/full_signal/scaling_exponents_acc.parquet', index=False)
    logger.info("Exponents saved.")
except Exception as e:
    logger.error(f"Error saving exponents: {e}")

INFO | Computing scaling exponents — acceleration, full signal...
INFO | Exponents saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


#### 3.1.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

In [6]:
logger.info("Linearity test and piecewise fit — acceleration, full signal...")
df_linearity_acc = test_scaling_linearity(
    df_exponents_acc, output_dir=FIGURES_DIR / 'scaling' / 'acceleration' / 'linearity')
try:
    df_linearity_acc.to_parquet('../data/processed/03b_moment_scaling/acceleration/full_signal/scaling_linearity_acc.parquet', index=False)
    logger.info("Linearity saved.")
except Exception as e:
    logger.error(f"Error saving linearity: {e}")

df_piecewise_acc = fit_piecewise_scaling(
    df_exponents_acc, output_dir=FIGURES_DIR / 'scaling' / 'acceleration' / 'piecewise')
try:
    df_piecewise_acc.to_parquet('../data/processed/03b_moment_scaling/acceleration/full_signal/scaling_piecewise_acc.parquet', index=False)
    logger.info("Piecewise saved.")
except Exception as e:
    logger.error(f"Error saving piecewise: {e}")

INFO | Linearity test and piecewise fit — acceleration, full signal...
INFO | Linearity saved.


Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    46
linear        2
Name: count, dtype: int64


INFO | Piecewise saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


### 3.2 Velocity

#### 3.2.1 Computation of q-th order moments

The process is defined as the ground velocity, obtained by integrating
the acceleration once:

$$v(t) = \sum_{k=0}^{t} a(k)\,\Delta t$$

Increments are computed as:

$$\Delta v(\tau, t_0) = v(t_0 + \tau) - v(t_0)$$

Moments $M_q(\tau)$ are computed for the same $q$ and $\tau$ values as
in Section 3.1.

In [7]:
logger.info("Integrating acceleration to velocity — full signal...")
df_acc_long_with_vel = integrate_to_velocity(df_acc_long, dt=0.005, normalized=False, method='trapz')
logger.info(f"Velocity range: [{df_acc_long_with_vel['velocity'].min():.4f}, "
            f"{df_acc_long_with_vel['velocity'].max():.4f}] cm/s")

logger.info("Computing velocity increments — full signal...")
q_values_vel = q_values
tau_values_vel = [10, 20, 35, 50, 75, 100, 150, 200, 300, 500,
                  750, 1000, 1500, 2000, 3000, 5000, 7500, 10000]

# Compute increments
df_increments_vel = compute_increments(df_acc_long_with_vel, tau_values_vel, column='velocity')
logger.info(f"Increments computed: {df_increments_vel.shape}")

# Save increments
try:
    df_increments_vel.to_parquet('../data/processed/03b_moment_scaling/velocity/full_signal/scaling_increments_vel.parquet', index=False)
    logger.info(f"Increments saved: {df_increments_vel.shape}")
except Exception as e:
    logger.error(f"Error saving increments: {e}")

# Compute moments
logger.info("Computing velocity moments — full signal...")
df_moments_vel = compute_moments_from_increments(df_increments_vel, q_values_vel)
logger.info(f"Moments computed: {df_moments_vel.shape}")

# Save moments
try:
    df_moments_vel.to_parquet('../data/processed/03b_moment_scaling/velocity/full_signal/scaling_moments_vel.parquet', index=False)
    logger.info(f"Moments saved: {df_moments_vel.shape}")
except Exception as e:
    logger.error(f"Error saving moments: {e}")

INFO | Integrating acceleration to velocity — full signal...
INFO | Velocity range: [-0.0458, 0.0361] cm/s
INFO | Computing velocity increments — full signal...
INFO | Increments computed: (40358880, 6)
INFO | Increments saved: (40358880, 6)
INFO | Computing velocity moments — full signal...
INFO | Moments computed: (16416, 6)
INFO | Moments saved: (16416, 6)


#### 3.2.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

In [8]:
logger.info("Computing scaling exponents — velocity, full signal...")
df_exponents_vel = compute_scaling_exponents(
    df_moments_vel, output_dir=FIGURES_DIR / 'scaling' / 'velocity' / 'exponents')
try:
    df_exponents_vel.to_parquet('../data/processed/03b_moment_scaling/velocity/full_signal/scaling_exponents_vel.parquet', index=False)
    logger.info("Exponents saved.")
except Exception as e:
    logger.error(f"Error saving exponents: {e}")

INFO | Computing scaling exponents — velocity, full signal...
INFO | Exponents saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


#### 3.2.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

In [9]:
logger.info("Linearity test and piecewise fit — velocity, full signal...")
df_linearity_vel = test_scaling_linearity(
    df_exponents_vel, output_dir=FIGURES_DIR / 'scaling' / 'velocity' / 'linearity')
try:
    df_linearity_vel.to_parquet('../data/processed/03b_moment_scaling/velocity/full_signal/scaling_linearity_vel.parquet', index=False)
    logger.info("Linearity saved.")
except Exception as e:
    logger.error(f"Error saving linearity: {e}")

df_piecewise_vel = fit_piecewise_scaling(
    df_exponents_vel, output_dir=FIGURES_DIR / 'scaling' / 'velocity' / 'piecewise')
try:
    df_piecewise_vel.to_parquet('../data/processed/03b_moment_scaling/velocity/full_signal/scaling_piecewise_vel.parquet', index=False)
    logger.info("Piecewise saved.")
except Exception as e:
    logger.error(f"Error saving piecewise: {e}")

INFO | Linearity test and piecewise fit — velocity, full signal...
INFO | Linearity saved.


Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    47
linear        1
Name: count, dtype: int64


INFO | Piecewise saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


### 3.3 Displacement

#### 3.3.1 Computation of q-th order moments

The process is defined as the ground displacement, obtained by integrating
the acceleration twice:

$$v(t) = \sum_{k=0}^{t} a(k)\,\Delta t, \qquad
x(t) = \sum_{k=0}^{t} v(k)\,\Delta t$$

Increments are computed as:

$$\Delta x(\tau, t_0) = x(t_0 + \tau) - x(t_0)$$

Moments $M_q(\tau)$ are computed for the same $q$ and $\tau$ values as
in Sections 3.1 and 3.2.

In [10]:
logger.info("Integrating acceleration to displacement — full signal...")
df_acc_long_with_disp = integrate_to_displacement(df_acc_long, dt=0.005, normalized=False, method='trapz')
logger.info(f"Displacement range: [{df_acc_long_with_disp['displacement'].min():.6f}, "
            f"{df_acc_long_with_disp['displacement'].max():.6f}] cm")

logger.info("Computing displacement increments — full signal...")
q_values_disp = q_values
tau_values_disp = [10, 20, 35, 50, 75, 100, 150, 200, 300, 500,
                   750, 1000, 1500, 2000, 3000, 5000, 7500, 10000]

# Compute increments
df_increments_disp = compute_increments(df_acc_long_with_disp, tau_values_disp, column='displacement')
logger.info(f"Increments computed: {df_increments_disp.shape}")

# Save increments
try:
    df_increments_disp.to_parquet('../data/processed/03b_moment_scaling/displacement/full_signal/scaling_increments_disp.parquet', index=False)
    logger.info(f"Increments saved: {df_increments_disp.shape}")
except Exception as e:
    logger.error(f"Error saving increments: {e}")

# Compute moments
logger.info("Computing displacement moments — full signal...")
df_moments_disp = compute_moments_from_increments(df_increments_disp, q_values_disp)
logger.info(f"Moments computed: {df_moments_disp.shape}")

# Save moments
try:
    df_moments_disp.to_parquet('../data/processed/03b_moment_scaling/displacement/full_signal/scaling_moments_disp.parquet', index=False)
    logger.info(f"Moments saved: {df_moments_disp.shape}")
except Exception as e:
    logger.error(f"Error saving moments: {e}")

INFO | Integrating acceleration to displacement — full signal...
INFO | Displacement range: [-0.002582, 0.004235] cm
INFO | Computing displacement increments — full signal...
INFO | Increments computed: (40358880, 6)
INFO | Increments saved: (40358880, 6)
INFO | Computing displacement moments — full signal...
INFO | Moments computed: (16416, 6)
INFO | Moments saved: (16416, 6)


#### 3.3.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

In [11]:
logger.info("Computing scaling exponents — displacement, full signal...")
df_exponents_disp = compute_scaling_exponents(
    df_moments_disp, output_dir=FIGURES_DIR / 'scaling' / 'displacement' / 'exponents')
try:
    df_exponents_disp.to_parquet('../data/processed/03b_moment_scaling/displacement/full_signal/scaling_exponents_disp.parquet', index=False)
    logger.info("Exponents saved.")
except Exception as e:
    logger.error(f"Error saving exponents: {e}")

INFO | Computing scaling exponents — displacement, full signal...
INFO | Exponents saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


#### 3.3.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

In [12]:
logger.info("Linearity test and piecewise fit — displacement, full signal...")
df_linearity_disp = test_scaling_linearity(
    df_exponents_disp, output_dir=FIGURES_DIR / 'scaling' / 'displacement' / 'linearity')
try:
    df_linearity_disp.to_parquet('../data/processed/03b_moment_scaling/displacement/full_signal/scaling_linearity_disp.parquet', index=False)
    logger.info("Linearity saved.")
except Exception as e:
    logger.error(f"Error saving linearity: {e}")

df_piecewise_disp = fit_piecewise_scaling(
    df_exponents_disp, output_dir=FIGURES_DIR / 'scaling' / 'displacement' / 'piecewise')
try:
    df_piecewise_disp.to_parquet('../data/processed/03b_moment_scaling/displacement/full_signal/scaling_piecewise_disp.parquet', index=False)
    logger.info("Piecewise saved.")
except Exception as e:
    logger.error(f"Error saving piecewise: {e}")

INFO | Linearity test and piecewise fit — displacement, full signal...
INFO | Linearity saved.


Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    48
Name: count, dtype: int64


INFO | Piecewise saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


### 3.4 Event window only

The analyses in Sections 3.1--3.3 use the full signal, including the
pre-event noise window (~12,000--15,000 samples) where the acceleration
is essentially zero. Here we repeat the displacement moment scaling
analysis restricting to the **event window**, defined as the portion of
the signal starting from the detected onset of the seismic waves.

The onset is detected as the first sample where $|\hat{a}(t)|$ exceeds
5% of the signal maximum for at least 10 consecutive samples.

#### Event onset detection

In [13]:
logger.info("Detecting event onsets...")
df_acc_event, df_onsets = trim_to_event_window(
    df_acc_long, threshold_factor=0.05, min_consecutive=10)
check(len(df_onsets) == df_acc_long['file'].nunique(),
      f"Onsets detected for all {len(df_onsets)} signals")
logger.info(f"Mean onset: {df_onsets['onset'].mean():.0f} samples — "
            f"Mean window length: {df_onsets['samples_after'].mean():.0f} samples")
print(df_onsets[['station', 'stream', 'onset', 'samples_before', 'samples_after']])
plot_onset_diagnostic(df_acc_long, df_onsets,
                      output_dir=FIGURES_DIR / 'scaling' / 'event_window')
plot_onset_distribution(df_onsets,
                        output_dir=FIGURES_DIR / 'scaling' / 'event_window')

INFO | Detecting event onsets...
INFO | Onsets detected for all 48 signals
INFO | Mean onset: 12396 samples — Mean window length: 36104 samples


   station stream  onset  samples_before  samples_after
0     EILF    HNE  12456           12456          35544
1     EILF    HNN  12164           12164          35836
2     EILF    HNZ  12190           12190          35810
3     ESCA    HNE  12414           12414          35586
4     ESCA    HNN  12402           12402          35598
5     ESCA    HNZ  12400           12400          35600
6      ISO    HNE  12578           12578          35422
7      ISO    HNN  12563           12563          35437
8      ISO    HNZ  12586           12586          35414
9      MFC    HNE  12350           12350          35650
10     MFC    HNN  12329           12329          35671
11     MFC    HNZ  12309           12309          35691
12     MON    HNE  12235           12235          35765
13     MON    HNN  12183           12183          35817
14     MON    HNZ  12188           12188          35812
15    MVIF    HNE  12394           12394          35606
16    MVIF    HNN  12348           12348        

Before computing moment scaling on the event window, we integrate acceleration to obtain velocity and displacement for all signals.

In [14]:
logger.info("=" * 80)
logger.info("PROCESS GENERATION - Event Window")
logger.info("=" * 80)

# Integrate acceleration to velocity and displacement
logger.info("Integrating acceleration to velocity and displacement...")
df_acc_event_with_processes = integrate_to_displacement(
    df_acc_event,
    dt=0.005,
    normalized=False,
    method='trapz'
)

# Validation checks
logger.info(f"\nShape: {df_acc_event_with_processes.shape}")
logger.info(f"Columns: {df_acc_event_with_processes.columns.tolist()}")
logger.info(f"Files: {df_acc_event_with_processes['file'].nunique()}")

logger.info(f"\nAcceleration range: [{df_acc_event_with_processes['acceleration'].min():.4f}, "
            f"{df_acc_event_with_processes['acceleration'].max():.4f}] cm/s²")
logger.info(f"Velocity range: [{df_acc_event_with_processes['velocity'].min():.4f}, "
            f"{df_acc_event_with_processes['velocity'].max():.4f}] cm/s")
logger.info(f"Displacement range: [{df_acc_event_with_processes['displacement'].min():.6f}, "
            f"{df_acc_event_with_processes['displacement'].max():.6f}] cm")

logger.info(f"\nNaN in acceleration: {df_acc_event_with_processes['acceleration'].isna().sum()}")
logger.info(f"NaN in velocity: {df_acc_event_with_processes['velocity'].isna().sum()}")
logger.info(f"NaN in displacement: {df_acc_event_with_processes['displacement'].isna().sum()}")

# Save processes
try:
    output_path = '../data/processed/event_window_with_processes.parquet'
    df_acc_event_with_processes.to_parquet(output_path, index=False)
    logger.info(f"\nSaved processes to: {output_path}")
except Exception as e:
    logger.error(f"Error saving processes: {e}")

INFO | ================================================================================
INFO | PROCESS GENERATION - Event Window
INFO | ================================================================================
INFO | Integrating acceleration to velocity and displacement...
INFO | 
Shape: (1733015, 5)
INFO | Columns: ['file', 'sample', 'acceleration', 'velocity', 'displacement']
INFO | Files: 48
INFO | 
Acceleration range: [-0.9768, 0.7344] cm/s²
INFO | Velocity range: [-0.0461, 0.0358] cm/s
INFO | Displacement range: [-0.463067, 1.855033] cm
INFO | 
NaN in acceleration: 0
INFO | NaN in velocity: 0
INFO | NaN in displacement: 0
INFO | 
Saved processes to: ../data/processed/event_window_with_processes.parquet


#### Moment scaling on event window

The moment scaling analysis is repeated on the trimmed signals using
the same $q$ and $\tau$ values as in Section 3.3. Note that the maximum
usable $\tau$ is constrained by the minimum event window length.

In [15]:
min_length = df_onsets['samples_after'].min()
tau_max = int(min_length / 3)
q_values_event  = q_values
tau_values_event = [t for t in [10, 20, 35, 50, 75, 100, 150, 200, 300,
                                 500, 750, 1000, 1500, 2000, 3000, 5000]
                    if t <= tau_max]
logger.info(f"Event window: min length = {min_length} samples — "
            f"tau_max = {tau_max} — {len(tau_values_event)} tau values")

INFO | Event window: min length = 35316 samples — tau_max = 11772 — 16 tau values


#### 3.4.1 Acceleration — event window only

In [16]:
logger.info("Computing acceleration increments — event window...")

# Compute increments
df_increments_acc_event = compute_increments(
    df_acc_event_with_processes, 
    tau_values_event, 
    column='acceleration'
)
logger.info(f"Increments computed: {df_increments_acc_event.shape}")

# Compute moments
logger.info("Computing acceleration moments — event window...")
df_moments_acc_event = compute_moments_from_increments(df_increments_acc_event, q_values_event)
logger.info(f"Moments computed: {df_moments_acc_event.shape}")

# Save
try:
    df_moments_acc_event.to_parquet('../data/processed/03b_moment_scaling/acceleration/event_window/scaling_moments_acc_event.parquet', index=False)
    df_increments_acc_event.to_parquet('../data/processed/03b_moment_scaling/acceleration/event_window/scaling_increments_acc_event.parquet', index=False)
    logger.info(f"Moments saved: {df_moments_acc_event.shape}")
    logger.info(f"Increments saved: {df_increments_acc_event.shape}")
except Exception as e:
    logger.error(f"Error saving: {e}")

# Compute scaling exponents
logger.info("Computing scaling exponents — acceleration, event window...")
df_exponents_acc_event = compute_scaling_exponents(
    df_moments_acc_event,
    output_dir=FIGURES_DIR / 'scaling' / 'acceleration' / 'event_window' / 'exponents')
try:
    df_exponents_acc_event.to_parquet('../data/processed/03b_moment_scaling/acceleration/event_window/scaling_exponents_acc_event.parquet', index=False)
    logger.info("Exponents saved.")
except Exception as e:
    logger.error(f"Error saving exponents: {e}")

# Linearity test and piecewise fit
logger.info("Linearity test and piecewise fit — acceleration, event window...")
df_linearity_acc_event = test_scaling_linearity(
    df_exponents_acc_event,
    output_dir=FIGURES_DIR / 'scaling' / 'acceleration' / 'event_window' / 'linearity')
try:
    df_linearity_acc_event.to_parquet('../data/processed/03b_moment_scaling/acceleration/event_window/scaling_linearity_acc_event.parquet', index=False)
    logger.info("Linearity saved.")
except Exception as e:
    logger.error(f"Error saving linearity: {e}")

df_piecewise_acc_event = fit_piecewise_scaling(
    df_exponents_acc_event,
    output_dir=FIGURES_DIR / 'scaling' / 'acceleration' / 'event_window' / 'piecewise')
try:
    df_piecewise_acc_event.to_parquet('../data/processed/03b_moment_scaling/acceleration/event_window/scaling_piecewise_acc_event.parquet', index=False)
    logger.info("Piecewise saved.")
except Exception as e:
    logger.error(f"Error saving piecewise: {e}")

INFO | Computing acceleration increments — event window...
INFO | Increments computed: (27023120, 6)
INFO | Computing acceleration moments — event window...
INFO | Moments computed: (14592, 6)
INFO | Moments saved: (14592, 6)
INFO | Increments saved: (27023120, 6)
INFO | Computing scaling exponents — acceleration, event window...
INFO | Exponents saved.
INFO | Linearity test and piecewise fit — acceleration, event window...


Saved: 48/48 individual plots
All individual plots saved successfully!


INFO | Linearity saved.


Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    47
linear        1
Name: count, dtype: int64


INFO | Piecewise saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


#### 3.4.2 Velocity — event window only

In [17]:
logger.info("Computing velocity increments — event window...")

# Compute increments
df_increments_vel_event = compute_increments(
    df_acc_event_with_processes, 
    tau_values_event, 
    column='velocity'
)
logger.info(f"Increments computed: {df_increments_vel_event.shape}")

# Compute moments
logger.info("Computing velocity moments — event window...")
df_moments_vel_event = compute_moments_from_increments(df_increments_vel_event, q_values_event)
logger.info(f"Moments computed: {df_moments_vel_event.shape}")

# Save
try:
    df_moments_vel_event.to_parquet('../data/processed/03b_moment_scaling/velocity/event_window/scaling_moments_vel_event.parquet', index=False)
    df_increments_vel_event.to_parquet('../data/processed/03b_moment_scaling/velocity/event_window/scaling_increments_vel_event.parquet', index=False)
    logger.info(f"Moments saved: {df_moments_vel_event.shape}")
    logger.info(f"Increments saved: {df_increments_vel_event.shape}")
except Exception as e:
    logger.error(f"Error saving: {e}")

# Compute scaling exponents
logger.info("Computing scaling exponents — velocity, event window...")
df_exponents_vel_event = compute_scaling_exponents(
    df_moments_vel_event,
    output_dir=FIGURES_DIR / 'scaling' / 'velocity' / 'event_window' / 'exponents')
try:
    df_exponents_vel_event.to_parquet('../data/processed/03b_moment_scaling/velocity/event_window/scaling_exponents_vel_event.parquet', index=False)
    logger.info("Exponents saved.")
except Exception as e:
    logger.error(f"Error saving exponents: {e}")

# Linearity test and piecewise fit
logger.info("Linearity test and piecewise fit — velocity, event window...")
df_linearity_vel_event = test_scaling_linearity(
    df_exponents_vel_event,
    output_dir=FIGURES_DIR / 'scaling' / 'velocity' / 'event_window' / 'linearity')
try:
    df_linearity_vel_event.to_parquet('../data/processed/03b_moment_scaling/velocity/event_window/scaling_linearity_vel_event.parquet', index=False)
    logger.info("Linearity saved.")
except Exception as e:
    logger.error(f"Error saving linearity: {e}")

df_piecewise_vel_event = fit_piecewise_scaling(
    df_exponents_vel_event,
    output_dir=FIGURES_DIR / 'scaling' / 'velocity' / 'event_window' / 'piecewise')
try:
    df_piecewise_vel_event.to_parquet('../data/processed/03b_moment_scaling/velocity/event_window/scaling_piecewise_vel_event.parquet', index=False)
    logger.info("Piecewise saved.")
except Exception as e:
    logger.error(f"Error saving piecewise: {e}")

INFO | Computing velocity increments — event window...
INFO | Increments computed: (27023120, 6)
INFO | Computing velocity moments — event window...
INFO | Moments computed: (14592, 6)
INFO | Moments saved: (14592, 6)
INFO | Increments saved: (27023120, 6)
INFO | Computing scaling exponents — velocity, event window...
INFO | Exponents saved.
INFO | Linearity test and piecewise fit — velocity, event window...


Saved: 48/48 individual plots
All individual plots saved successfully!


INFO | Linearity saved.


Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    46
linear        2
Name: count, dtype: int64


INFO | Piecewise saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


#### 3.4.3 Displacement - event window only

In [18]:
logger.info("Computing displacement increments — event window...")

# Compute increments
df_increments_disp_event = compute_increments(
    df_acc_event_with_processes, 
    tau_values_event, 
    column='displacement'
)
logger.info(f"Increments computed: {df_increments_disp_event.shape}")

# Compute moments
logger.info("Computing displacement moments — event window...")
df_moments_disp_event = compute_moments_from_increments(df_increments_disp_event, q_values_event)
logger.info(f"Moments computed: {df_moments_disp_event.shape}")

# Save
try:
    df_moments_disp_event.to_parquet('../data/processed/03b_moment_scaling/displacement/event_window/scaling_moments_disp_event.parquet', index=False)
    df_increments_disp_event.to_parquet('../data/processed/03b_moment_scaling/displacement/event_window/scaling_increments_disp_event.parquet', index=False)
    logger.info(f"Moments saved: {df_moments_disp_event.shape}")
    logger.info(f"Increments saved: {df_increments_disp_event.shape}")
except Exception as e:
    logger.error(f"Error saving: {e}")

# Compute scaling exponents
logger.info("Computing scaling exponents — displacement, event window...")
df_exponents_disp_event = compute_scaling_exponents(
    df_moments_disp_event,
    output_dir=FIGURES_DIR / 'scaling' / 'displacement' / 'event_window' / 'exponents')
try:
    df_exponents_disp_event.to_parquet('../data/processed/03b_moment_scaling/displacement/event_window/scaling_exponents_disp_event.parquet', index=False)
    logger.info("Exponents saved.")
except Exception as e:
    logger.error(f"Error saving exponents: {e}")

# Linearity test and piecewise fit
logger.info("Linearity test and piecewise fit — displacement, event window...")
df_linearity_disp_event = test_scaling_linearity(
    df_exponents_disp_event,
    output_dir=FIGURES_DIR / 'scaling' / 'displacement' / 'event_window' / 'linearity')
try:
    df_linearity_disp_event.to_parquet('../data/processed/03b_moment_scaling/displacement/event_window/scaling_linearity_disp_event.parquet', index=False)
    logger.info("Linearity saved.")
except Exception as e:
    logger.error(f"Error saving linearity: {e}")

df_piecewise_disp_event = fit_piecewise_scaling(
    df_exponents_disp_event,
    output_dir=FIGURES_DIR / 'scaling' / 'displacement' / 'event_window' / 'piecewise')
try:
    df_piecewise_disp_event.to_parquet('../data/processed/03b_moment_scaling/displacement/event_window/scaling_piecewise_disp_event.parquet', index=False)
    logger.info("Piecewise saved.")
except Exception as e:
    logger.error(f"Error saving piecewise: {e}")

INFO | Computing displacement increments — event window...
INFO | Increments computed: (27023120, 6)
INFO | Computing displacement moments — event window...
INFO | Moments computed: (14592, 6)
INFO | Moments saved: (14592, 6)
INFO | Increments saved: (27023120, 6)
INFO | Computing scaling exponents — displacement, event window...
INFO | Exponents saved.
INFO | Linearity test and piecewise fit — displacement, event window...


Saved: 48/48 individual plots
All individual plots saved successfully!


INFO | Linearity saved.


Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    48
Name: count, dtype: int64


INFO | Piecewise saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


#### Increment distributions and tail exponents

The empirical CCDF of $|\Delta x(\tau)|$ is plotted in log-log scale for
six representative time scales $\tau \in \{10, 100, 500, 1000, 3000, 5000\}$
samples. The tail exponent is estimated independently via the Hill estimator
and via a linear fit on the log-log CCDF (top 10% of values).

## 4. Summary

A summary table collects the main results from all analyses for each signal.
For the moment scaling columns, results from the displacement version (Section 3.3)
are reported as the physically most meaningful choice.

| Column | Description |
|--------|-------------|
| `kurtosis` | Excess kurtosis $\kappa$ |
| `skewness` | Skewness $\gamma$ |
| `non_gaussian` | Anderson-Darling test result ($\alpha = 0.05$) |
| `best_fit_aic` | Best fitting distribution by AIC |
| `student_t_df` | Student-$t$ degrees of freedom $\nu$ |
| `power_law_exp` | Power law exponent $\hat{\alpha}$ (Hill estimator) |
| `q_break` | Piecewise scaling breakpoint $q^*$ |
| `slope_low` | Scaling slope for $q \leq q^*$ |
| `slope_high` | Scaling slope for $q > q^*$ |
| `beta` | Autocorrelation scaling exponent $\beta$ |


### 4.1 Acceleration

In [19]:
df_pdf_summary = pd.read_parquet('../data/processed/03a_pdf_analysis/pdf_summary.parquet')

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/03a_pdf_analysis/pdf_summary.parquet'

In [ ]:
logger.info("Building acceleration summary...")
df_summary_acc = build_scaling_summary(
    df_exponents_acc_event, 
    df_piecewise_acc_event, 
    'acc'
)

### 4.2 Velocity

In [ ]:
logger.info("Building velocity summary...")
df_summary_vel = build_scaling_summary(
    df_exponents_vel_event, 
    df_piecewise_vel_event, 
    'vel'
)

### 4.3 Displacement

In [ ]:
logger.info("Building displacement summary...")
df_summary_disp = build_scaling_summary(
    df_exponents_disp_event, 
    df_piecewise_disp_event, 
    'disp'
)

In [ ]:
# ---------------------------------------------------------------------------
# 4.3 Merge all summaries
# ---------------------------------------------------------------------------

# Start with PDF summary
df_summary = df_pdf_summary.copy()

# Merge moment scaling summaries
df_summary = df_summary.merge(df_summary_acc, on='file', how='left')
df_summary = df_summary.merge(df_summary_vel, on='file', how='left')
df_summary = df_summary.merge(df_summary_disp, on='file', how='left')

# Optional: Add metadata if available
# df_summary = df_summary.merge(df_metadata[['file', 'component', 'distance_km', 'pga_cms2']], 
#                               on='file', how='left')

# ---------------------------------------------------------------------------
# 4.4 Reorder columns for readability
# ---------------------------------------------------------------------------

# Group columns by analysis type
pdf_cols = ['file', 'kurtosis', 'skewness', 'non_gaussian', 'best_fit_aic', 
            'student_t_df', 'power_law_exp']

acc_cols = [col for col in df_summary.columns if '_acc' in col]
vel_cols = [col for col in df_summary.columns if '_vel' in col]
disp_cols = [col for col in df_summary.columns if '_disp' in col]

# Autocorrelation column (if it exists)
autocorr_cols = ['beta'] if 'beta' in df_summary.columns else []

# Reorder
column_order = pdf_cols + acc_cols + vel_cols + disp_cols + autocorr_cols
df_summary = df_summary[column_order]

# ---------------------------------------------------------------------------
# 4.5 Display and save
# ---------------------------------------------------------------------------

logger.info(f"Summary table built: {df_summary.shape}")
display(df_summary.head(10))

# Save
try:
    df_summary.to_parquet('../data/processed/complete_summary.parquet', index=False)
    logger.info("Summary saved: complete_summary.parquet")
except Exception as e:
    logger.error(f"Error saving summary: {e}")

# ---------------------------------------------------------------------------
# 4.6 Summary statistics across all signals
# ---------------------------------------------------------------------------

logger.info("\n" + "="*80)
logger.info("SUMMARY STATISTICS - Event Window Results")
logger.info("="*80)

for process in ['acc', 'vel', 'disp']:
    logger.info(f"\n{process.upper()}:")
    logger.info(f"  ζ(q=2): {df_summary[f'zeta_q2_{process}'].mean():.3f} ± {df_summary[f'zeta_q2_{process}'].std():.3f}")
    logger.info(f"  ζ(q=4): {df_summary[f'zeta_q4_{process}'].mean():.3f} ± {df_summary[f'zeta_q4_{process}'].std():.3f}")
    logger.info(f"  q*:     {df_summary[f'q_break_{process}'].mean():.2f} ± {df_summary[f'q_break_{process}'].std():.2f}")
    logger.info(f"  α₁:     {df_summary[f'slope_low_{process}'].mean():.3f} ± {df_summary[f'slope_low_{process}'].std():.3f}")
    logger.info(f"  α₂:     {df_summary[f'slope_high_{process}'].mean():.3f} ± {df_summary[f'slope_high_{process}'].std():.3f}")

logger.info("\n" + "="*80)